In [18]:
"""
Six charts for legislator audience: three per sample (full renter sample and
income-restricted subset).

Per sample:
  1. Prevalence overview: horizontal bar chart showing the weighted % of
     renter households experiencing each of nine specific housing problems,
     ordered by prevalence (most common at top).

  2. Racial composition: 100%-stacked horizontal bar chart with one bar per
     problem PLUS a reference bar showing the racial composition of the
     full sample. Bars in same order as chart 1.

  3. Income/subsidy composition: 100%-stacked horizontal bar chart breaking
     each problem down by five categories: Public Housing, Voucher, Other
     subsidy, Unsubsidized low-income, and Non-low-income. (In the
     eligible-only sample, the Non-low-income category is excluded by the
     eligibility filter and so is dropped automatically.)

Samples:
  A. ALL: every occupied renter household (CSR in {5, 32, 80, 90, 97}).
  B. ELIGIBLE: subset earning <=80% AMI for the 2023 NYC HMFA OR receiving
     any rental assistance (RENTASSIST == 1). This mirrors the eligibility
     filter used in the regression analysis.

Inputs (raw NYCHVS PUF subsets):
  /mnt/user-data/uploads/all_units_dataset_subset.csv
  /mnt/user-data/uploads/occupied_dataset_subset.csv
  /mnt/user-data/uploads/person_dataset_subset.csv

Outputs:
  /mnt/user-data/outputs/problem_prevalence_all.png
  /mnt/user-data/outputs/problem_racial_composition_all.png
  /mnt/user-data/outputs/problem_tier_composition_all.png
  /mnt/user-data/outputs/problem_prevalence_eligible.png
  /mnt/user-data/outputs/problem_racial_composition_eligible.png
  /mnt/user-data/outputs/problem_tier_composition_eligible.png
  /mnt/user-data/outputs/tier_by_race.png   (standalone: 5 tier bars stacked
                                              by race, full sample only)

Notes:
  * Race assigned from the household respondent (RELATION_P == 0) using
    RACEETH_P, with categories 5 (NHPI/AIAN) and 6 (Other/2+) merged into
    "Other / 2+" since the individual cells are too small to display.
  * Income/subsidy tier: every renter falls into exactly one of:
      - Public Housing: CSR == 5 (NYCHA).
      - Voucher: RENTASSIST_VOUCHER == 1 and not in public housing.
      - Other subsidy: RENTASSIST == 1 but neither public housing nor voucher
        (e.g., SCRIE, DRIE, City FHEPS).
      - Unsubsidized low-income: income-eligible but no subsidy.
      - Non-low-income: not eligible (income > 80% AMI AND no rental
        assistance).
  * Negative codes are treated as MISSING for each problem indicator
    (excluded from BOTH numerator and denominator) -- EXCEPT for the
    skip-pattern -2 codes on NOHEAT_NUM and PEELPAINT_LARGE, which mean
    "no problem" (skipped because the parent question already said no) and
    are coded as FALSE.
  * Weighted estimates use FW (housing-unit final survey weight). Replicate
    weights are not used here -- this is descriptive and the SDR machinery
    isn't needed for stacked-bar tabulations at presentation precision.
  * Style follows the NYC Council `councildown` package (Georgia font with
    serif fallback; 'main' palette; clean theme: no gridlines, no ticks,
    light-grey axis lines).
"""


'\nSix charts for legislator audience: three per sample (full renter sample and\nincome-restricted subset).\n\nPer sample:\n  1. Prevalence overview: horizontal bar chart showing the weighted % of\n     renter households experiencing each of nine specific housing problems,\n     ordered by prevalence (most common at top).\n\n  2. Racial composition: 100%-stacked horizontal bar chart with one bar per\n     problem PLUS a reference bar showing the racial composition of the\n     full sample. Bars in same order as chart 1.\n\n  3. Income/subsidy composition: 100%-stacked horizontal bar chart breaking\n     each problem down by five categories: Public Housing, Voucher, Other\n     subsidy, Unsubsidized low-income, and Non-low-income. (In the\n     eligible-only sample, the Non-low-income category is excluded by the\n     eligibility filter and so is dropped automatically.)\n\nSamples:\n  A. ALL: every occupied renter household (CSR in {5, 32, 80, 90, 97}).\n  B. ELIGIBLE: subset earning <=

In [19]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

In [20]:
INPUT_DIR  = "../data/processed"
OUT_PREV_PATTERN  = "../visuals/problem_prevalence_{sample}.png"
OUT_STACK_PATTERN = "../visuals/problem_racial_composition_{sample}.png"
OUT_TIER_PATTERN  = "../visuals/problem_tier_composition_{sample}.png"
OUT_TIER_BY_RACE  = "../visuals/tier_by_race.png"
OUT_TIER_BY_RACE_ELIGIBLE  = "../visuals/tier_by_race_eligible.png"

In [21]:
# ---------------------------------------------------------------------------
# NYCC style (mirrors theme_nycc.R + colors.R)
# ---------------------------------------------------------------------------
_AVAILABLE_FONTS = {f.name for f in fm.fontManager.ttflist}
if   "Georgia" in _AVAILABLE_FONTS:        NYCC_FONT = "Georgia"
elif "DejaVu Serif" in _AVAILABLE_FONTS:   NYCC_FONT = "DejaVu Serif"
else:                                       NYCC_FONT = "serif"
matplotlib.rcParams["font.family"] = NYCC_FONT

NYCC_AXIS_GREY = "#CACACA"
NYCC_DARK_GREY = "#666666"
# 'main' categorical palette from colors.R (≤7 colors)
NYCC_MAIN = ["#660000", "#1850b5", "#ba9f64",
             "#1f3a70", "#b3b3ff", "#af6d46", "#666666"]
# Single-color emphasis for the prevalence chart
NYCC_BLUE = "#2F56A6"

def _apply_nycc_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_color(NYCC_AXIS_GREY)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.spines["left"].set_color(NYCC_AXIS_GREY)
    ax.spines["left"].set_linewidth(0.8)
    ax.tick_params(axis="both", which="both", length=0)
    ax.grid(False)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontsize(11); lbl.set_fontfamily(NYCC_FONT)

In [22]:
# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------
all_units = pd.read_csv(f"{INPUT_DIR}/all_units_dataset_subset.csv")
occupied  = pd.read_csv(f"{INPUT_DIR}/occupied_dataset_subset.csv")
person    = pd.read_csv(f"{INPUT_DIR}/person_dataset_subset.csv")

# Restrict all_units to occupied renter units
RENTER_CSR = [5, 32, 80, 90, 97]
renters = all_units[(all_units["OCC"] == 1)
                    & (all_units["CSR"].isin(RENTER_CSR))].copy()
df = renters.merge(occupied.drop(columns=["FW"]), on="CONTROL", how="inner")

# Attach respondent race
resp = person.loc[person["RELATION_P"] == 0,
                  ["CONTROL", "RACEETH_P"]].copy()
df = df.merge(resp, on="CONTROL", how="left")

RACE_LABELS = {
    1: "White", 2: "Black", 3: "Hispanic", 4: "Asian",
    5: "Other / 2+", 6: "Other / 2+",
}
df["race"] = df["RACEETH_P"].map(RACE_LABELS)

print(f"Renter sample: {len(df):,} occupied renter households")
print("Race distribution (unweighted):")
print(df["race"].value_counts().to_string())

Renter sample: 6,659 occupied renter households
Race distribution (unweighted):
race
Hispanic      2155
White         2076
Black         1529
Asian          801
Other / 2+      98


In [23]:
# ---------------------------------------------------------------------------
# Build problem indicators (TRUE = problem; NaN = missing/not reported)
# ---------------------------------------------------------------------------
def coded_true(series, true_vals, missing_vals=(-3, -2, -1)):
    """
    Return a boolean Series where True = value in true_vals, False = a clean
    "no problem" code, NaN = anything in missing_vals.
    """
    out = pd.Series(np.nan, index=series.index, dtype="object")
    out[series.isin(missing_vals)] = np.nan
    out[series.isin(true_vals)] = True
    out[~series.isin(missing_vals) & ~series.isin(true_vals)] = False
    return out

# 1. Heating breakdown (1+ of 6+ hours)
#    NOHEAT_NUM == -2 means "no heating breakdowns at all" (skip pattern from
#    NOHEAT==2), so it counts as FALSE, not missing.
df["p_noheat"] = pd.Series(
    np.where(df["NOHEAT_NUM"].isin([-3, -1]), np.nan,
             df["NOHEAT_NUM"].isin([1, 2, 3, 4])),
    index=df.index, dtype="object",
)
# 2. No hot water
df["p_nohotwater"] = coded_true(df["NOHOTWATER"], true_vals=[1])
# 3. Leaks
df["p_leaks"]     = coded_true(df["LEAKS"], true_vals=[1])
# 4. Mold or musty smell
mold_true = df["MOLD"].isin([1])
mold_miss = df["MOLD"].isin([-3, -2, -1])
musty_true = df["MUSTY"].isin([1, 2, 3, 4])
musty_miss = df["MUSTY"].isin([-3, -2, -1])
both_miss = mold_miss & musty_miss
df["p_mold_musty"] = pd.Series(np.where(both_miss, np.nan,
                                         (mold_true | musty_true)),
                                index=df.index, dtype="object")
# 5. Rodents (in unit OR in building)
ru_true = df["RODENTS_UNIT"].isin([1])
ru_miss = df["RODENTS_UNIT"].isin([-3, -2, -1])
rb_true = df["RODENTS_BUILD"].isin([1])
rb_miss = df["RODENTS_BUILD"].isin([-3, -2, -1])
both_rmiss = ru_miss & rb_miss
df["p_rodents"] = pd.Series(np.where(both_rmiss, np.nan,
                                      (ru_true | rb_true)),
                              index=df.index, dtype="object")
# 6. Broken toilet
df["p_toilet"] = coded_true(df["TOILET_BROK"], true_vals=[1])
# 7. Holes in walls or floors
wh_true = df["WALLHOLES"].isin([1])
wh_miss = df["WALLHOLES"].isin([-3, -2, -1])
fh_true = df["FLOORHOLES"].isin([1])
fh_miss = df["FLOORHOLES"].isin([-3, -2, -1])
both_hmiss = wh_miss & fh_miss
df["p_holes"] = pd.Series(np.where(both_hmiss, np.nan,
                                    (wh_true | fh_true)),
                            index=df.index, dtype="object")
# 8. Large peeling paint
#    PEELPAINT_LARGE == -2 means "no peeling paint at all" (skip pattern from
#    PEELPAINT==2), so it counts as FALSE, not missing.
df["p_peel_large"] = pd.Series(
    np.where(df["PEELPAINT_LARGE"].isin([-3, -1]), np.nan,
             df["PEELPAINT_LARGE"] == 1),
    index=df.index, dtype="object",
)
# 9. Roaches (any seen)
df["p_roaches"] = coded_true(df["ROACHES_NUM"], true_vals=[2, 3, 4])

# Rollup: "one or more of the above issues."
# Hybrid rule:
#   - True  if ANY problem indicator is True (positive evidence of >=1 problem
#     wins regardless of missingness on other indicators).
#   - False if ALL problem indicators are False (clean negative on all 9).
#   - NaN   if no indicator is True and at least one is missing (we can't
#     conclude the household has zero problems when we don't have data on all
#     of them).
_problem_cols = ["p_noheat", "p_nohotwater", "p_leaks", "p_mold_musty",
                 "p_rodents", "p_toilet", "p_holes", "p_peel_large", "p_roaches"]
_problem_mat = df[_problem_cols]   # values are True / False / NaN
_any_true   = _problem_mat.eq(True).any(axis=1)
_any_miss   = _problem_mat.isna().any(axis=1)
df["p_any"] = pd.Series(
    np.where(_any_true, True,
             np.where(_any_miss, np.nan, False)),
    index=df.index, dtype="object",
)

PROBLEMS = [
    ("p_noheat",      "Heating outage (1+ breakdown ≥6 hrs)"),
    ("p_nohotwater",  "Hot water outage"),
    ("p_leaks",       "Leaks"),
    ("p_mold_musty",  "Mold or musty smell"),
    ("p_rodents",     "Rodents in unit or building"),
    ("p_toilet",      "Broken toilet"),
    ("p_holes",       "Holes in walls or floors"),
    ("p_peel_large",  "Large sections of peeling paint"),
    ("p_roaches",     "Roaches seen in last month"),
]


In [24]:
# ---------------------------------------------------------------------------
# Income-eligibility filter (mirrors build_howell_replication_dataset.py)
#   Eligible = household income <= 80% AMI for NYC HMFA by HHSIZE,
#              OR receiving any rental assistance (RENTASSIST == 1).
# ---------------------------------------------------------------------------
AMI80_2023_NYC_HMFA = {
    1: 79200, 2: 90500, 3: 101800, 4: 113100,
    5: 122150, 6: 131200, 7: 140250, 8: 149300,
}

def income_limit_for_hhsize(n: int) -> float:
    if n <= 8:
        return AMI80_2023_NYC_HMFA[n]
    return AMI80_2023_NYC_HMFA[8] + (n - 8) * 0.08 * AMI80_2023_NYC_HMFA[4]

df["INC_LIMIT_80AMI"] = df["HHSIZE"].clip(lower=1).apply(income_limit_for_hhsize)
df["INCOME_ELIGIBLE"] = (df["HHINC_REC1"] <= df["INC_LIMIT_80AMI"]).astype(int)
df["ANY_RENTASSIST"]  = (df["RENTASSIST"] == 1).astype(int)
df["IS_ELIGIBLE"]     = ((df["INCOME_ELIGIBLE"] == 1)
                          | (df["ANY_RENTASSIST"] == 1)).astype(bool)

# 5-way income/subsidy tier (every renter falls into exactly one):
#   1. Public Housing:           CSR == 5 (NYCHA)
#   2. Voucher:                  RENTASSIST_VOUCHER == 1 (Section 8 housing
#                                choice voucher) AND CSR != 5
#   3. Other subsidy:            RENTASSIST == 1 but neither public housing
#                                nor voucher (e.g., SCRIE, DRIE, City FHEPS)
#   4. Unsubsidized low-income:  income-eligible but no subsidy
#   5. Non-low-income:           not eligible (income > 80% AMI AND no
#                                rental assistance)
def assign_tier(row):
    if row["CSR"] == 5:
        return "Public Housing"
    if row["RENTASSIST_VOUCHER"] == 1:
        return "Voucher"
    if row["RENTASSIST"] == 1:
        return "Other subsidy"
    if row["IS_ELIGIBLE"]:
        return "Unsubsidized low-income"
    return "Non-low-income"
df["INCOME_TIER"] = df.apply(assign_tier, axis=1)

print(f"\nEligibility breakdown:")
print(f"  Total renter sample:         {len(df):,}")
print(f"  Income-eligible (<=80% AMI): {(df['INCOME_ELIGIBLE']==1).sum():,}")
print(f"  Receiving rental assistance: {(df['ANY_RENTASSIST']==1).sum():,}")
print(f"  Eligible (income OR assist): {df['IS_ELIGIBLE'].sum():,}")
print(f"\n5-way income/subsidy tier breakdown:")
print(df["INCOME_TIER"].value_counts().to_string())


Eligibility breakdown:
  Total renter sample:         6,659
  Income-eligible (<=80% AMI): 4,341
  Receiving rental assistance: 1,173
  Eligible (income OR assist): 4,397

5-way income/subsidy tier breakdown:
INCOME_TIER
Unsubsidized low-income    2761
Non-low-income             2232
Public Housing              664
Voucher                     562
Other subsidy               440


In [25]:
# ---------------------------------------------------------------------------
# Compute weighted prevalence per problem (per-sample helper)
# ---------------------------------------------------------------------------
def weighted_prevalence(data, col, weight_col="FW"):
    """Weighted % True among non-missing rows; also return weighted N affected."""
    mask = data[col].notna()
    sub = data.loc[mask, [col, weight_col]].copy()
    sub[col] = sub[col].astype(bool)
    total_w = sub[weight_col].sum()
    aff_w   = sub.loc[sub[col], weight_col].sum()
    return aff_w / total_w, aff_w, total_w, int(mask.sum())


ANY_ROLLUP_LABEL = "One or more of the above issues"


def build_prevalence_table(data):
    rows = []
    for col, label in PROBLEMS:
        pct, aff_w, tot_w, n_used = weighted_prevalence(data, col)
        rows.append({"var": col, "label": label, "prevalence": pct,
                     "weighted_affected": aff_w, "weighted_universe": tot_w,
                     "n_used": n_used})
    table = (pd.DataFrame(rows)
                .sort_values("prevalence", ascending=False)
                .reset_index(drop=True))

    # Append rollup last, AFTER sorting, so it always sits at the end of the
    # prevalence chart regardless of its actual value (which will be the
    # highest of any bar -- ~70-80%).
    any_pct, any_w, any_tot_w, any_n = weighted_prevalence(data, "p_any")
    rollup = pd.DataFrame([{
        "var": "p_any", "label": ANY_ROLLUP_LABEL, "prevalence": any_pct,
        "weighted_affected": any_w, "weighted_universe": any_tot_w,
        "n_used": any_n,
    }])
    return pd.concat([table, rollup], ignore_index=True)

In [26]:
# ---------------------------------------------------------------------------
# Race / palette setup (shared across charts)
# ---------------------------------------------------------------------------
RACE_ORDER  = ["White", "Black", "Hispanic", "Asian", "Other / 2+"]
RACE_COLORS = {
    "White":       NYCC_MAIN[6],   # grey -- White as a neutral reference color
    "Black":       NYCC_MAIN[0],   # maroon
    "Hispanic":    NYCC_MAIN[1],   # blue
    "Asian":       NYCC_MAIN[2],   # bronze
    "Other / 2+":  NYCC_MAIN[5],   # salmon-brown
}

# Income/subsidy tier categorization. Five mutually exclusive categories,
# ordered for stacking left-to-right: Public Housing -> Voucher -> Other
# subsidy -> Unsubsidized low-income -> Non-low-income (most policy-controlled
# at left, least at right).
TIER_ORDER  = [
    "Public Housing",
    "Voucher",
    "Other subsidy",
    "Unsubsidized low-income",
    "Non-low-income",
]
TIER_COLORS = {
    "Public Housing":          NYCC_MAIN[0],   # maroon -- strongest emphasis
                                                # (most directly policy-controlled)
    "Voucher":                 NYCC_MAIN[1],   # blue -- second policy-prominent
    "Other subsidy":           NYCC_MAIN[2],   # bronze
    "Unsubsidized low-income": NYCC_MAIN[5],   # salmon-brown
    "Non-low-income":          NYCC_MAIN[3],   # dark navy -- "above threshold"
}

def category_shares(data, cat_col, cat_order, weight_col="FW"):
    """Return weighted share per category value within a (sub)sample.

    Generic: works for any categorical column (e.g., 'race', 'INCOME_TIER').
    Rows with NaN in cat_col are dropped from the share computation.
    """
    sub = data.dropna(subset=[cat_col])
    tot = sub[weight_col].sum()
    if tot == 0:
        return {c: 0.0 for c in cat_order}
    return {c: sub.loc[sub[cat_col] == c, weight_col].sum() / tot
            for c in cat_order}


def format_pop(weighted_count):
    """Format a weighted population estimate for display.
       1,234,567 -> '1.2M'; 12,345 -> '12K'; 1,234,000 -> '1.2M'.
       Honest about precision: weighted survey estimates have noise, so don't
       pretend to be exact down to the household."""
    if weighted_count >= 1_000_000:
        return f"{weighted_count/1_000_000:.1f}M"
    if weighted_count >= 1_000:
        return f"{int(round(weighted_count/1_000)):,}K"
    return f"{int(round(weighted_count)):,}"

In [27]:
# ---------------------------------------------------------------------------
# Chart 1 builder: prevalence overview
# ---------------------------------------------------------------------------
def build_prevalence_chart(data, prev_df, out_path, *,
                            denom_label, n_total, weighted_total):
    """Render the horizontal-bar prevalence chart for one sample."""
    fig, ax = plt.subplots(figsize=(10, 6.0))

    # Split prev_df: specific-problem rows (all but the last) and the rollup
    # (last row, by construction in build_prevalence_table). We want the chart
    # to show, top-to-bottom:
    #   1. Specific problems sorted DESC by prevalence (most common at top)
    #   2. "One or more of the above issues" rollup at the very bottom
    spec_rows  = prev_df.iloc[:-1]
    # any_row    = prev_df.iloc[-1:]
    # matplotlib's y-axis runs bottom-to-top, so to render specific problems
    # in descending order from the top we put them in ASC order in the array,
    # then put the rollup at index 0 (which renders at the bottom).
    # plot_df = pd.concat([any_row, spec_rows.iloc[::-1]],
    #                      ignore_index=True)
    plot_df = spec_rows.iloc[::-1]
    y = np.arange(len(plot_df))
    ax.barh(y, plot_df["prevalence"] * 100, color=NYCC_BLUE, edgecolor="none")

    for i, p in enumerate(plot_df["prevalence"]):
        ax.text(p * 100 + 0.5, i, f"{p*100:.1f}%",
                va="center", fontsize=11, family=NYCC_FONT, color="#222222")

    # Dashed separator above the rollup bar to mark it as a roll-up rather
    # than a 10th specific problem. The rollup is at index 0 (bottom), so the
    # line goes at y=0.5 (between rollup and the lowest specific problem).
    # ax.axhline(0.5, color=NYCC_DARK_GREY, linewidth=0.6, linestyle="--",
    #            alpha=0.7)

    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["label"], fontsize=11, family=NYCC_FONT)
    ax.set_xlabel(f"Percent of {denom_label}",
                  fontsize=12, family=NYCC_FONT, labelpad=10)
    ax.set_xlim(0, max(plot_df["prevalence"] * 100) * 1.2)
    ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
        lambda v, _: f"{int(v)}%"))

    fig.suptitle(f"Prevalence of housing problems among {denom_label}",
                 fontsize=16, family=NYCC_FONT, color="#222222",
                 x=0.02, ha="left", y=0.98)
    fig.text(0.02, 0.93,
             f"Survey n = {n_total:,} respondents, representing "
             f"≈ {format_pop(weighted_total)} households "
             "(2023 NYCHVS, weighted estimates)",
             fontsize=12, family=NYCC_FONT, color=NYCC_DARK_GREY,
             ha="left", va="top")
    fig.text(0.02, 0.01,
             "Specific problem bars sorted by prevalence."
             "Negative response codes excluded "
             "from both numerator and denominator.",
             fontsize=9, family=NYCC_FONT, color=NYCC_DARK_GREY,
             ha="left", va="bottom")

    _apply_nycc_axes(ax)
    plt.tight_layout(rect=[0.02, 0.04, 1, 0.88])
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Wrote {out_path}")

In [28]:
# ---------------------------------------------------------------------------
# Chart 2 builder: 100% stacked racial composition with reference bar
# ---------------------------------------------------------------------------
def build_stacked_chart(data, prev_df, out_path, *, denom_label,
                         cat_col, cat_order, cat_colors,
                         dimension_label, title_text):
    """Render a 100%-stacked horizontal bar chart of any categorical breakdown.

    Parameters
    ----------
    cat_col          : column to break down by (e.g., 'race', 'INCOME_TIER')
    cat_order        : list of category values in the order they should stack
                       (left to right)
    cat_colors       : mapping {category value -> hex color}
    dimension_label  : human-readable name of the dimension, used in the
                       footer (e.g., 'race', 'income/subsidy')
    title_text       : text for the chart title; '{denom_label}' will be
                       substituted with denom_label
    """
    ref_shares = category_shares(data, cat_col, cat_order)
    bar_rows = []
    for _, row in prev_df.iterrows():
        col = row["var"]; label = row["label"]
        aff = data[data[col] == True]
        # Drop NaN on the breakdown column so the weighted total aligns with
        # the slices being stacked.
        aff_with_cat = aff.dropna(subset=[cat_col])
        bar_rows.append({"label": label,
                         "shares": category_shares(aff, cat_col, cat_order),
                         "n_unw": len(aff_with_cat),
                         "weighted": aff_with_cat["FW"].sum()})

    ref_label = f"All {denom_label} (reference)"
    ref_n_unw = len(data.dropna(subset=[cat_col]))
    ref_weighted = data.dropna(subset=[cat_col])["FW"].sum()

    # Plot order top-to-bottom (before reversing for matplotlib):
    #   1. Reference bar
    #   2. Per-problem bars (sorted by prevalence in prev_df)
    #   3. "One or more" rollup -- baked into prev_df as the last row, so it
    #      arrives last via the iteration above and ends up at the bottom of
    #      the rendered chart.
    plot_labels = [ref_label] + [r["label"] for r in bar_rows]
    plot_shares = [ref_shares] + [r["shares"] for r in bar_rows]
    plot_n      = [ref_n_unw] + [r["n_unw"] for r in bar_rows]
    plot_pop    = [ref_weighted] + [r["weighted"] for r in bar_rows]

    # Reverse for matplotlib so top of plot = first item
    plot_labels = plot_labels[::-1]
    plot_shares = plot_shares[::-1]
    plot_n      = plot_n[::-1]
    plot_pop    = plot_pop[::-1]

    fig, ax = plt.subplots(figsize=(11, 6.5))
    y = np.arange(len(plot_labels))

    left = np.zeros(len(plot_labels))
    for cat in cat_order:
        vals = np.array([s[cat] for s in plot_shares])
        ax.barh(y, vals, left=left, color=cat_colors[cat],
                edgecolor="white", linewidth=0.8, label=cat)
        for i, (v, l) in enumerate(zip(vals, left)):
            if v >= 0.04:
                ax.text(l + v / 2, i, f"{v*100:.0f}%",
                        ha="center", va="center", fontsize=9,
                        family=NYCC_FONT, color="white", fontweight="bold")
        left += vals

    # Dashed separators: one below the reference bar (separating baseline
    # from data), one above the rollup bar (separating per-problem bars from
    # the all-problems summary). Both serve the same role: marking a
    # discontinuity in what each bar represents.
    ref_idx = plot_labels.index(ref_label)
    ax.axhline(ref_idx - 0.5, color=NYCC_DARK_GREY, linewidth=0.6,
               linestyle="--", alpha=0.7)
    rollup_idx = plot_labels.index(ANY_ROLLUP_LABEL)
    ax.axhline(rollup_idx + 0.5, color=NYCC_DARK_GREY, linewidth=0.6,
               linestyle="--", alpha=0.7)

    ax.set_yticks(y)
    ax.set_yticklabels(plot_labels, fontsize=11, family=NYCC_FONT)
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"],
                       fontsize=11, family=NYCC_FONT)
    ax.set_xlabel("Share of households in each category",
                  fontsize=12, family=NYCC_FONT, labelpad=10)

    # Right-side annotations: weighted population estimate for each bar
    for i, pop in enumerate(plot_pop):
        ax.text(1.01, i, f"≈ {format_pop(pop)} households",
                va="center", ha="left", fontsize=9,
                family=NYCC_FONT, color=NYCC_DARK_GREY)

    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12),
              ncol=len(cat_order), frameon=False,
              prop={"family": NYCC_FONT, "size": 10})

    fig.suptitle(
        title_text.format(denom_label=denom_label),
        fontsize=16, family=NYCC_FONT, color="#222222",
        x=0.02, ha="left", y=0.99,
    )
    fig.text(0.02, 0.93,
             f"Survey n = {ref_n_unw:,} respondents, representing "
             f"≈ {format_pop(ref_weighted)} households. Compare each problem "
             "bar to the reference bar.",
             fontsize=12, family=NYCC_FONT, color=NYCC_DARK_GREY,
             ha="left", va="top")
    # fig.text(0.02, 0.01,
    #          "Specific problem bars are ordered by overall prevalence (most "
    #          "common first). 'One or more' is a rollup across all 9 problems. "
    #          f"Segments show the weighted % of households in each "
    #          f"{dimension_label} group within the subset.",
    #          fontsize=9, family=NYCC_FONT, color=NYCC_DARK_GREY,
    #          ha="left", va="bottom")

    _apply_nycc_axes(ax)
    plt.tight_layout(rect=[0.02, 0.06, 0.92, 0.88])
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Wrote {out_path}")

In [29]:
# ---------------------------------------------------------------------------
# Chart 4 builder: 100% stacked tier-by-race composition
#   One bar per income/subsidy tier, stacked by race, plus a reference bar
#   showing the racial composition of all NYC renters.
#
# Distinct from build_stacked_chart -- that one's bars are defined by
# membership in `data[col] == True` for problem indicators. Here, bars are
# defined by membership in `data["INCOME_TIER"] == tier_value` for each tier.
# Same NYCC styling, same reference-bar pattern, no rollup row.
# ---------------------------------------------------------------------------
def build_tier_by_race_chart(data, out_path, *,
                              tier_col, tier_order,
                              cat_col, cat_order, cat_colors,
                              denom_label, dimension_label, title_text):
    """Render a 100%-stacked horizontal bar chart with one bar per tier.

    Parameters
    ----------
    tier_col         : column whose values define the bars (e.g., 'INCOME_TIER')
    tier_order       : list of tier values, in the y-axis order to display
                       (top-to-bottom)
    cat_col          : column whose values define the stack segments
                       (e.g., 'race')
    cat_order        : list of segment values left-to-right
    cat_colors       : mapping {category value -> hex color}
    denom_label      : human-readable name of the population (e.g., 'NYC renters')
    dimension_label  : name of the stack dimension, used in the footer
                       (e.g., 'race')
    title_text       : chart title, with '{denom_label}' substitution available
    """
    # Reference bar: racial composition of the full sample
    ref_shares = category_shares(data, cat_col, cat_order)
    ref_label = f"All {denom_label} (reference)"
    ref_n_unw = len(data.dropna(subset=[cat_col]))
    ref_weighted = data.dropna(subset=[cat_col])["FW"].sum()

    # One bar per tier
    bar_rows = []
    for tier in tier_order:
        sub = data[data[tier_col] == tier]
        sub_with_cat = sub.dropna(subset=[cat_col])
        bar_rows.append({"label": tier,
                         "shares": category_shares(sub, cat_col, cat_order),
                         "n_unw": len(sub_with_cat),
                         "weighted": sub_with_cat["FW"].sum()})

    # Plot order top-to-bottom: reference, then tiers in tier_order
    plot_labels = [ref_label] + [r["label"] for r in bar_rows]
    plot_shares = [ref_shares] + [r["shares"] for r in bar_rows]
    plot_n      = [ref_n_unw] + [r["n_unw"] for r in bar_rows]
    plot_pop    = [ref_weighted] + [r["weighted"] for r in bar_rows]

    # Reverse for matplotlib so top of plot = first item
    plot_labels = plot_labels[::-1]
    plot_shares = plot_shares[::-1]
    plot_n      = plot_n[::-1]
    plot_pop    = plot_pop[::-1]

    fig, ax = plt.subplots(figsize=(11, 5.5))
    y = np.arange(len(plot_labels))

    left = np.zeros(len(plot_labels))
    for cat in cat_order:
        vals = np.array([s[cat] for s in plot_shares])
        ax.barh(y, vals, left=left, color=cat_colors[cat],
                edgecolor="white", linewidth=0.8, label=cat)
        for i, (v, l) in enumerate(zip(vals, left)):
            if v >= 0.04:
                ax.text(l + v / 2, i, f"{v*100:.0f}%",
                        ha="center", va="center", fontsize=9,
                        family=NYCC_FONT, color="white", fontweight="bold")
        left += vals

    # Single dashed separator below the reference bar (no rollup, so no
    # second separator at the bottom).
    ref_idx = plot_labels.index(ref_label)
    ax.axhline(ref_idx - 0.5, color=NYCC_DARK_GREY, linewidth=0.6,
               linestyle="--", alpha=0.7)

    ax.set_yticks(y)
    ax.set_yticklabels(plot_labels, fontsize=11, family=NYCC_FONT)
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"],
                       fontsize=11, family=NYCC_FONT)
    ax.set_xlabel("Share of households in each category",
                  fontsize=12, family=NYCC_FONT, labelpad=10)

    for i, pop in enumerate(plot_pop):
        ax.text(1.01, i, f"≈ {format_pop(pop)} households",
                va="center", ha="left", fontsize=9,
                family=NYCC_FONT, color=NYCC_DARK_GREY)

    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25),
              ncol=len(cat_order), frameon=False,
              prop={"family": NYCC_FONT, "size": 10})

    fig.suptitle(
        title_text.format(denom_label=denom_label),
        fontsize=16, family=NYCC_FONT, color="#222222",
        x=0.02, ha="left", y=0.99,
    )
    fig.text(0.02, 0.90,
             f"Survey n = {ref_n_unw:,} respondents, representing "
             f"≈ {format_pop(ref_weighted)} households. Compare each tier bar "
             "to the reference bar.",
             fontsize=12, family=NYCC_FONT, color=NYCC_DARK_GREY,
             ha="left", va="top")
    # fig.text(0.02, 0.01,
    #          "Tiers ordered by program type (most policy-controlled first). "
    #          f"Segments show the weighted % of households in each "
    #          f"{dimension_label} group within each tier.",
    #          fontsize=9, family=NYCC_FONT, color=NYCC_DARK_GREY,
    #          ha="left", va="bottom", wrap=True)

    _apply_nycc_axes(ax)
    plt.tight_layout(rect=[0.02, 0.12, 0.92, 0.85])
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Wrote {out_path}")

In [30]:
# ---------------------------------------------------------------------------
# Driver: build all six charts (three per sample) plus the tier-by-race chart
# ---------------------------------------------------------------------------
SAMPLE_DEFS = [
    ("all",      df,                       "NYC renters"),
    ("eligible", df[df["IS_ELIGIBLE"]],    "income-eligible NYC renters"),
]

for sample_key, sample_data, denom_label in SAMPLE_DEFS:
    print(f"\n--- Sample: {sample_key} (n={len(sample_data):,}) ---")
    prev_df = build_prevalence_table(sample_data)
    print("Weighted prevalence per problem (sorted desc):")
    print(prev_df[["label", "prevalence", "n_used"]]
          .assign(prevalence=lambda x: (x["prevalence"]*100).round(1).astype(str)+"%")
          .to_string(index=False))

    weighted_total = sample_data["FW"].sum()
    build_prevalence_chart(
        sample_data, prev_df,
        OUT_PREV_PATTERN.format(sample=sample_key),
        denom_label=denom_label, n_total=len(sample_data),
        weighted_total=weighted_total,
    )

    # Race-composition chart
    build_stacked_chart(
        sample_data, prev_df,
        OUT_STACK_PATTERN.format(sample=sample_key),
        denom_label=denom_label,
        cat_col="race", cat_order=RACE_ORDER, cat_colors=RACE_COLORS,
        dimension_label="race",
        title_text="Racial composition by housing problem ({denom_label})",
    )

    # Income/subsidy-tier composition chart.
    # In the eligible-only sample, the "Non-low-income" category is empty by
    # definition, so we drop it from the order to avoid a zero-width slice.
    tiers_present = [t for t in TIER_ORDER
                      if (sample_data["INCOME_TIER"] == t).any()]
    tier_colors_present = {t: TIER_COLORS[t] for t in tiers_present}
    build_stacked_chart(
        sample_data, prev_df,
        OUT_TIER_PATTERN.format(sample=sample_key),
        denom_label=denom_label,
        cat_col="INCOME_TIER", cat_order=tiers_present,
        cat_colors=tier_colors_present,
        dimension_label="income/subsidy",
        title_text="Income and subsidy status by housing problem ({denom_label})",
    )


--- Sample: all (n=6,659) ---
Weighted prevalence per problem (sorted desc):
                               label prevalence  n_used
          Roaches seen in last month      37.2%    6617
         Rodents in unit or building      32.1%    6465
                               Leaks      24.8%    5796
                 Mold or musty smell      23.8%    5781
            Holes in walls or floors      21.8%    6659
Heating outage (1+ breakdown ≥6 hrs)      18.4%    5855
                    Hot water outage      16.2%    5833
     Large sections of peeling paint       9.4%    6659
                       Broken toilet       7.3%    6451
     One or more of the above issues      73.0%    6172
Wrote ../visuals/problem_prevalence_all.png
Wrote ../visuals/problem_racial_composition_all.png
Wrote ../visuals/problem_tier_composition_all.png

--- Sample: eligible (n=4,397) ---
Weighted prevalence per problem (sorted desc):
                               label prevalence  n_used
          Roaches see

In [31]:
df[(df["IS_ELIGIBLE"]) & (df['INCOME_TIER'] != 'Non-low-income')]


,CONTROL,CSR,OCC,YEARBUILT,UNITS,ROOMS,BORO,FW,FW1_x,FW2_x,...,p_toilet,p_holes,p_peel_large,p_roaches,p_any,INC_LIMIT_80AMI,INCOME_ELIGIBLE,ANY_RENTASSIST,IS_ELIGIBLE,INCOME_TIER
2,12130010,80,1,4,10,1,3,226.590865,226.590865,219.345692,...,False,0.0,0.0,True,1.0,79200.0,1,0,True,Unsubsidized low-income
4,12130018,32,1,4,6,3,4,509.268761,509.268761,506.606040,...,False,0.0,0.0,True,1.0,79200.0,1,1,True,Other subsidy
5,12130024,80,1,11,7,1,1,511.852389,511.852389,510.270627,...,False,1.0,0.0,False,1.0,79200.0,1,0,True,Unsubsidized low-income
6,12130028,5,1,5,10,4,1,192.451639,192.451639,202.305429,...,False,0.0,1.0,True,1.0,101800.0,1,0,True,Public Housing
8,12130034,97,1,7,9,4,2,81.347247,81.347247,130.526117,...,False,0.0,1.0,False,1.0,79200.0,1,1,True,Voucher
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6650,12158868,32,1,4,9,3,1,203.962884,203.962884,344.482941,...,False,0.0,0.0,True,1.0,79200.0,1,1,True,Voucher
6652,12158874,97,1,6,10,3,2,107.223401,107.223401,31.075647,...,False,0.0,0.0,False,1.0,79200.0,1,1,True,Other subsidy
6653,12158877,32,1,5,8,4,2,231.330588,231.330588,224.791855,...,False,1.0,0.0,True,1.0,79200.0,1,0,True,Unsubsidized low-income
6655,12158886,32,1,10,9,2,1,256.429010,256.429010,75.962910,...,False,0.0,0.0,False,1.0,90500.0,1,1,True,Voucher


In [32]:
TIER_ORDER[0:4]

['Public Housing', 'Voucher', 'Other subsidy', 'Unsubsidized low-income']

In [33]:
# ---------------------------------------------------------------------------
# Standalone chart: racial composition within each income/subsidy tier.
# Run once on the full renter sample (eligible-only would drop Non-low-income
# and reduce to 4 bars, which conveys less).
# ---------------------------------------------------------------------------
print("\n--- Standalone: tier-by-race chart ---")
build_tier_by_race_chart(
    df[(df["IS_ELIGIBLE"]) & (df['INCOME_TIER'] != 'Non-low-income')], 
    OUT_TIER_BY_RACE_ELIGIBLE,
    tier_col="INCOME_TIER", tier_order=TIER_ORDER[0:4],
    cat_col="race", cat_order=RACE_ORDER, cat_colors=RACE_COLORS,
    denom_label="income-eligible NYC renters",
    dimension_label="race",
    title_text="Racial composition within each income/subsidy tier ({denom_label})",
)


--- Standalone: tier-by-race chart ---
Wrote ../visuals/tier_by_race_eligible.png


In [34]:
"""
Histograms of the count of housing problems per household.

Each household has a count from 0 to 9 of how many of these distinct types of
housing problems they reported:
  1. Heating outage (1+ breakdown >=6 hours)
  2. Hot water outage
  3. Leaks
  4. Mold or musty smell
  5. Rodents in unit or building
  6. Broken toilet
  7. Holes in walls or floors
  8. Large sections of peeling paint
  9. Roaches seen in last month

Two separate figures (one per sample):
  - All NYC renters
  - Income-eligible NYC renters (<=80% AMI for NYC HMFA OR receiving rental
    assistance)

Each figure stands on its own with its own y-axis. The x-axis goes 0-9 in both.

Sample handling: households with missing data on ANY indicator are dropped.
The count needs to be exact for a histogram axis to be meaningful. Reporting
the dropped fraction in the footer.

Inputs (raw NYCHVS PUF subsets):
  /mnt/user-data/uploads/all_units_dataset_subset.csv
  /mnt/user-data/uploads/occupied_dataset_subset.csv

Outputs:
  /mnt/user-data/outputs/problem_count_histogram_all.png
  /mnt/user-data/outputs/problem_count_histogram_eligible.png

Style follows the NYC Council `councildown` package.
"""

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

INPUT_DIR = "../data/processed"
OUT_PATTERN = "../visuals/problem_count_histogram_{sample}.png"

# ---------------------------------------------------------------------------
# NYCC style
# ---------------------------------------------------------------------------
_AVAILABLE_FONTS = {f.name for f in fm.fontManager.ttflist}
if   "Georgia" in _AVAILABLE_FONTS:        NYCC_FONT = "Georgia"
elif "DejaVu Serif" in _AVAILABLE_FONTS:   NYCC_FONT = "DejaVu Serif"
else:                                       NYCC_FONT = "serif"
matplotlib.rcParams["font.family"] = NYCC_FONT

NYCC_AXIS_GREY = "#CACACA"
NYCC_DARK_GREY = "#666666"
NYCC_BLUE      = "#2F56A6"

def _apply_nycc_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_color(NYCC_AXIS_GREY)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.spines["left"].set_color(NYCC_AXIS_GREY)
    ax.spines["left"].set_linewidth(0.8)
    ax.tick_params(axis="both", which="both", length=0)
    ax.grid(False)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontsize(11); lbl.set_fontfamily(NYCC_FONT)

# ---------------------------------------------------------------------------
# Load and merge
# ---------------------------------------------------------------------------
all_units = pd.read_csv(f"{INPUT_DIR}/all_units_dataset_subset.csv")
occupied  = pd.read_csv(f"{INPUT_DIR}/occupied_dataset_subset.csv")

RENTER_CSR = [5, 32, 80, 90, 97]
renters = all_units[(all_units["OCC"] == 1)
                    & (all_units["CSR"].isin(RENTER_CSR))].copy()
df = renters.merge(occupied.drop(columns=["FW"]), on="CONTROL", how="inner")
print(f"Renter sample: {len(df):,} occupied renter households")

# ---------------------------------------------------------------------------
# Build the 9 problem indicators (same logic as problem_prevalence_charts.py)
# ---------------------------------------------------------------------------
def coded_true(series, true_vals, missing_vals=(-3, -2, -1)):
    out = pd.Series(np.nan, index=series.index, dtype="object")
    out[series.isin(missing_vals)] = np.nan
    out[series.isin(true_vals)] = True
    out[~series.isin(missing_vals) & ~series.isin(true_vals)] = False
    return out

# 1. Heating: NOHEAT_NUM == -2 is skip-pattern False, not missing
df["p_noheat"] = pd.Series(
    np.where(df["NOHEAT_NUM"].isin([-3, -1]), np.nan,
             df["NOHEAT_NUM"].isin([1, 2, 3, 4])),
    index=df.index, dtype="object",
)
df["p_nohotwater"] = coded_true(df["NOHOTWATER"], true_vals=[1])
df["p_leaks"]      = coded_true(df["LEAKS"], true_vals=[1])
mold_t = df["MOLD"].isin([1]); mold_m = df["MOLD"].isin([-3,-2,-1])
musty_t = df["MUSTY"].isin([1,2,3,4]); musty_m = df["MUSTY"].isin([-3,-2,-1])
df["p_mold_musty"] = pd.Series(
    np.where(mold_m & musty_m, np.nan, (mold_t | musty_t)),
    index=df.index, dtype="object",
)
ru_t = df["RODENTS_UNIT"].isin([1]); ru_m = df["RODENTS_UNIT"].isin([-3,-2,-1])
rb_t = df["RODENTS_BUILD"].isin([1]); rb_m = df["RODENTS_BUILD"].isin([-3,-2,-1])
df["p_rodents"] = pd.Series(
    np.where(ru_m & rb_m, np.nan, (ru_t | rb_t)),
    index=df.index, dtype="object",
)
df["p_toilet"] = coded_true(df["TOILET_BROK"], true_vals=[1])
wh_t = df["WALLHOLES"].isin([1]); wh_m = df["WALLHOLES"].isin([-3,-2,-1])
fh_t = df["FLOORHOLES"].isin([1]); fh_m = df["FLOORHOLES"].isin([-3,-2,-1])
df["p_holes"] = pd.Series(
    np.where(wh_m & fh_m, np.nan, (wh_t | fh_t)),
    index=df.index, dtype="object",
)
# PEELPAINT_LARGE == -2 is skip-pattern False
df["p_peel_large"] = pd.Series(
    np.where(df["PEELPAINT_LARGE"].isin([-3, -1]), np.nan,
             df["PEELPAINT_LARGE"] == 1),
    index=df.index, dtype="object",
)
df["p_roaches"] = coded_true(df["ROACHES_NUM"], true_vals=[2, 3, 4])

PROBLEM_COLS = ["p_noheat", "p_nohotwater", "p_leaks", "p_mold_musty",
                "p_rodents", "p_toilet", "p_holes", "p_peel_large", "p_roaches"]

# Compute the per-household problem count (strict: NaN if any indicator missing)
problem_mat = df[PROBLEM_COLS]
df["any_missing"] = problem_mat.isna().any(axis=1)
# Sum True values, treating False as 0; result is a count 0..9 when no NaN
df["problem_count"] = problem_mat.eq(True).sum(axis=1).where(~df["any_missing"])

n_total       = len(df)
n_complete    = (~df["any_missing"]).sum()
print(f"Households with complete data on all 9 indicators: {n_complete:,} "
      f"({n_complete/n_total*100:.1f}% of renter sample)")

# ---------------------------------------------------------------------------
# Income-eligibility filter (mirrors build script)
# ---------------------------------------------------------------------------
AMI80_2023_NYC_HMFA = {
    1: 79200, 2: 90500, 3: 101800, 4: 113100,
    5: 122150, 6: 131200, 7: 140250, 8: 149300,
}
def income_limit_for_hhsize(n: int) -> float:
    if n <= 8:
        return AMI80_2023_NYC_HMFA[n]
    return AMI80_2023_NYC_HMFA[8] + (n - 8) * 0.08 * AMI80_2023_NYC_HMFA[4]

df["INC_LIMIT_80AMI"] = df["HHSIZE"].clip(lower=1).apply(income_limit_for_hhsize)
df["IS_ELIGIBLE"] = ((df["HHINC_REC1"] <= df["INC_LIMIT_80AMI"])
                      | (df["RENTASSIST"] == 1))

# ---------------------------------------------------------------------------
# Build weighted histograms for both samples
# ---------------------------------------------------------------------------
def weighted_histogram(data, count_col="problem_count", weight_col="FW",
                        max_count=9):
    """Return weighted population at each count value 0..max_count."""
    sub = data.dropna(subset=[count_col]).copy()
    sub[count_col] = sub[count_col].astype(int)
    counts = np.zeros(max_count + 1)
    for k in range(max_count + 1):
        counts[k] = sub.loc[sub[count_col] == k, weight_col].sum()
    return counts


def format_pop(weighted_count):
    if weighted_count >= 1_000_000:
        return f"{weighted_count/1_000_000:.1f}M"
    if weighted_count >= 1_000:
        return f"{int(round(weighted_count/1_000)):,}K"
    return f"{int(round(weighted_count)):,}"


SAMPLES = [
    ("all",      "All NYC renters",            df),
    ("eligible", "Income-eligible NYC renters", df[df["IS_ELIGIBLE"]]),
]

# ---------------------------------------------------------------------------
# Plot one figure per sample
# ---------------------------------------------------------------------------
x = np.arange(10)
for sample_key, sample_label, sample_data in SAMPLES:
    weighted = weighted_histogram(sample_data)
    n_unw_complete = (~sample_data["any_missing"]).sum()
    n_unw_total    = len(sample_data)
    n_unw_dropped  = sample_data["any_missing"].sum()
    weighted_total = weighted.sum()
    y_cap          = weighted.max() * 1.18

    fig, ax = plt.subplots(figsize=(10, 6))

    bars = ax.bar(x, weighted, color=NYCC_BLUE, edgecolor="none", width=0.8)

    # Value labels above each non-empty bar
    for k, v in enumerate(weighted):
        if v <= 0:
            continue
        ax.text(k, v + y_cap * 0.015, format_pop(v),
                ha="center", va="bottom",
                fontsize=11, family=NYCC_FONT, color="#222222")

    ax.set_ylim(0, y_cap)
    ax.set_xticks(x)
    ax.set_xticklabels([str(i) for i in x], fontsize=11, family=NYCC_FONT)
    ax.set_xlabel("Number of housing problems",
                  fontsize=12, family=NYCC_FONT, labelpad=10)
    ax.set_ylabel("Households (weighted)",
                  fontsize=12, family=NYCC_FONT, labelpad=10)
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
        lambda v, _: format_pop(v) if v > 0 else "0"))

    fig.suptitle(
        f"Number of housing problems per household — {sample_label}",
        fontsize=16, family=NYCC_FONT, color="#222222",
        x=0.02, ha="left", y=0.98,
    )
    fig.text(
        0.02, 0.93,
        f"Survey n = {n_unw_complete:,} respondents (of {n_unw_total:,}), "
        f"representing ≈ {format_pop(weighted_total)} households "
        "(2023 NYCHVS, weighted estimates)",
        fontsize=12, family=NYCC_FONT, color=NYCC_DARK_GREY,
        ha="left", va="top",
    )
    fig.text(
        0.02, 0.01,
        f"Households with missing data on any of the 9 indicators are excluded "
        f"({n_unw_dropped:,} of {n_unw_total:,}). An exact household-level "
        "count requires complete data on all indicators.",
        fontsize=9, family=NYCC_FONT, color=NYCC_DARK_GREY,
        ha="left", va="bottom",
    )

    _apply_nycc_axes(ax)
    out_path = OUT_PATTERN.format(sample=sample_key)
    plt.tight_layout(rect=[0.02, 0.05, 1, 0.88])
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Wrote {out_path}")

# Also print the underlying numbers
print("\nWeighted distribution by problem count:")
table = pd.DataFrame({
    "count":            list(range(10)),
    "All renters":      [int(round(v)) for v in
                          weighted_histogram(SAMPLES[0][2])],
    "Eligible renters": [int(round(v)) for v in
                          weighted_histogram(SAMPLES[1][2])],
})
print(table.to_string(index=False))

Renter sample: 6,659 occupied renter households
Households with complete data on all 9 indicators: 5,723 (85.9% of renter sample)
Wrote ../visuals/problem_count_histogram_all.png
Wrote ../visuals/problem_count_histogram_eligible.png

Weighted distribution by problem count:
 count  All renters  Eligible renters
     0       577821            332888
     1       436603            278794
     2       324701            217761
     3       230196            156771
     4       155860            112760
     5       109075             84818
     6        69271             55620
     7        43281             34234
     8        19629             16472
     9         5877              4711
